In [ ]:
from dotenv import load_dotenv
from langchain_core.messages import HumanMessage
from langchain_deepseek import ChatDeepSeek
from langgraph.checkpoint.postgres import PostgresSaver
from langgraph.graph import MessagesState, StateGraph, START, END
import os

load_dotenv(override=True)
from rich import print

model = ChatDeepSeek(
    model="deepseek-v4-flash",
    extra_body=
    {
        "thinking": {
            "type": "disabled"
        }
    }
)


#1. 状態を宣言
class OverAllState(MessagesState):
    output: str


#2. ノードを宣言
def llm_mode(state: OverAllState) -> OverAllState:
    messages = state["messages"]
    res = model.invoke(messages)
    return {
        "messages": [res]
    }


def output_node(state: OverAllState) -> OverAllState:
    return {
        "output": state["messages"][-1].content
    }


#3. グラフを構築
builder = StateGraph(state_schema=OverAllState)
builder.add_node("llm_node", llm_mode)
builder.add_node("output_node", output_node)
builder.add_edge(START, "llm_node")
builder.add_edge("llm_node", "output_node")
builder.add_edge("output_node", END)

#4. チェックポイントストレージを設定
DB_URL = os.getenv("DB_URL")

with PostgresSaver.from_conn_string(DB_URL) as checkpointer:
    #5. PostgresSaver を初めてチェックポイントとして使う場合は setup() の呼び出しが必要
    checkpointer.setup()
    graph = builder.compile(checkpointer=checkpointer)

    # スレッドIDを指定
    config = {
        "configurable": {
            "thread_id": "chapter03-02"
        }
    }
    # グラフを呼び出す
    # graph.invoke({"messages":[HumanMessage("こんにちは、私は田中です")]},config=config)
    res = graph.invoke({"messages": [HumanMessage("最初の質問は？")]}, config=config)
    print(res)



In [10]:
with PostgresSaver.from_conn_string(DB_URL) as checkpointer:
    #5. PostgresSaver を初めてチェックポイントとして使う場合は setup() の呼び出しが必要
    # checkpointer.setup()
    graph = builder.compile(checkpointer=checkpointer)

    # スレッドIDを指定
    config = {
        "configurable": {
            "thread_id": "chapter03-02"
        }
    }
    # グラフを呼び出す
    # graph.invoke({"messages":[HumanMessage("こんにちは、私は田中です")]},config=config)
    res = graph.invoke({"messages": [HumanMessage("私の名前は?")]}, config=config)
    print(res)

{
    'messages': [
        HumanMessage(
            content='こんにちは、私は田中です',
            additional_kwargs={},
            response_metadata={},
            id='28a20129-fb22-4046-8005-9cf76ba8919d'
        ),
        AIMessage(
            content='こんにちは、田中さん！お会いできてうれしいです。  
\n何かお手伝いできることがあれば、いつでもお知らせくださいね。',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 40,
                    'prompt_tokens': 13,
                    'total_tokens': 53,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0},
                    'prompt_cache_hit_tokens': 0,
                    'prompt_cache_miss_tokens': 13
                },
                'model_provider': 'deepseek',
                'model_name': 'deepseek-v4-flash',
                'system_fingerprint': 'a26a7955944dc5c60445bff77fac9c8e',
                'id': 'f49c860c-df75-4910-8bc4-311430a759c3',
                'finish_reason': 'stop',
                'logprobs': None
            },
            id='lc_run--01a00033-bd45-7411-a878-b17711d3023f-0',
            tool_calls=[],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 13,
                'output_tokens': 40,
                'total_tokens': 53,
                'input_token_details': {'cache_read': 0},
                'output_token_details': {}
            }
        ),
        HumanMessage(
            content='最初の質問は？',
            additional_kwargs={},
            response_metadata={},
            id='e3054289-5498-4d0f-8101-958863c24f0e'
        ),
        AIMessage(
            content='いい質問ですね！  
\nまだ具体的な質問はいただいていなかったので、こちらからいくつか例を挙げますね。  
\n\n例えばこんな質問はいかがですか？  \n\n1. **日本語の学習**について（文法や表現の確認など）  \n2. 
**旅行**について（おすすめの場所や計画の立て方）  \n3. **テクノロジー**について（AIやプログラミングの基本など）  
\n4. **日常のちょっとした疑問**（料理、健康、趣味など）  \n\nもちろん、これ以外でも大丈夫です。  
\n田中さんが気になっていることを、ぜひ教えてください！😊',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 140,
                    'prompt_tokens': 62,
                    'total_tokens': 202,
                    'completion_tokens_details': None,
                    'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0},
                    'prompt_cache_hit_tokens': 0,
                    'prompt_cache_miss_tokens': 62
                },
                'model_provider': 'deepseek',
                'model_name': 'deepseek-v4-flash',
                'system_fingerprint': 'a26a7955944dc5c60445bff77fac9c8e',
                'id': '03ef3b4d-7472-4a82-8ea6-c284b677a625',
                'finish_reason': 'stop',
                'logprobs': None
            },
            id='lc_run--01a00034-0a86-7573-8c3b-3d888b200bea-0',
            tool_calls=[],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 62,
                'output_tokens': 140,
                'total_tokens': 202,
                'input_token_details': {'cache_read': 0},
                'output_token_details': {}
            }
        ),
        HumanMessage(
            content='第一个问题是什么?',
            additional_kwargs={},
            response_metadata={},
            id='a1f96722-cf4b-4c7d-a15c-90f8d9fc5c75'
        ),
        AIMessage(
            content='あ、すみません！私の説明が足りなかったですね。  
\n\n「最初の質問」というのは、**田中さんが私に投げかけたい質問**のことです。  
\nまだ田中さんから具体的な質問を受け取っていないので、「どんなことでも聞いてくださいね」という意味で、**質問の例**
をいくつか挙げただけでした。  \n\nつまり、**これから田中さんがする最初の質問**が、まさに「最初の質問」になります。 
\n何でもお気軽にどうぞ！😊  \n（もし「私が質問する番ですよ」という意味でしたら、遠慮なくどうぞ！）',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 142,
                    'prompt_tokens': 210,
                    'total_tokens': 352,
                    'compl